# Week 2 — Spiking simulation and sanity checks

Uses the weight set from `02_weights.ipynb` and the LIF module `sim/lif.py` (Shiu et al. 2024 parameters, copied from their `model.py`).

Checks, in order (each must pass before the next):
1. **Direct targets:** stimulate JO-C/E; their direct postsynaptic partners should fire.
2. Sugar GRNs → MN9 *(next)*
3. JO-C/E → descending candidates → front-leg motor neurons *(next)*
4. Shuffled control *(next)*

Memory: WSL is capped at 4 GB, so the checks run on subnetworks. Each build prints peak memory.

### Setup

In [ ]:
import sys
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

BRAIN = Path.home() / 'arm_simulation/brain'     # on Colab, change only these two lines
DATA = BRAIN / 'data'
sys.path.insert(0, str(BRAIN))

from sim.lif import WeightSet, LIFNet, SHIU, peak_mem_gb
from brian2 import Hz, second, prefs

ws = WeightSet(DATA / 'weights_male-cns_v1.0')
print(f"{ws.N:,} neurons, {len(ws.count):,} edges, guessed neurons: {ws.meta['guessed_neurons']:,}")
print(f"Brian2 code generation: {prefs.codegen.target}")
print(f"peak memory: {peak_mem_gb():.2f} GB")

### Cell 1: check the histamine fill

Step 2 filled 5,854 unclear neurons as histamine. That's fine if they are photoreceptor types; if non-photoreceptor types show up here, tell me.

In [ ]:
h = ws.neurons[ws.neurons.guessed & (ws.neurons.nt_used == 'histamine')]
print(f"guessed as histamine: {len(h):,}")
display(h['cls'].value_counts().to_frame('neurons'))
display(h['type'].value_counts().head(15).to_frame('neurons'))

### Cell 2: neuron groups

JO-C/E and the Week 1 shortlist, plus a search for the check 2 neurons (sugar GRNs and MN9). Send me the search output; the exact male-CNS type names decide check 2.

In [ ]:
JO = ws.by_type(r'^JO-(C|E)', cls='cb_sensory')
print(f"JO-C/E: {len(JO)} neurons (Week 1: 335)")

shortlist = pd.read_csv(DATA / 'shortlist_week1.csv')
g2i = pd.Series(ws.neurons.index, index=ws.neurons.bodyId)
DN_cand = g2i.loc[shortlist.query("group == 'middle_descending'").bodyId].to_numpy()
LEG_MN  = g2i.loc[shortlist.query("group == 'output_T1_leg_MN'").bodyId].to_numpy()
print(f"descending candidates: {len(DN_cand)}  front-leg motor neurons: {len(LEG_MN)}")

types = ws.neurons['type'].astype(str)
sens = ws.neurons[(ws.neurons.cls == 'cb_sensory') &
                  types.str.contains(r'(?i)gr5|gr64|sugar|sweet|gust|GRN|taste|pharyn|labell')]
print(f"\ngustatory-looking sensory neurons: {len(sens)}")
display(sens['type'].value_counts().head(30).to_frame('neurons'))

mn = ws.neurons[types.str.contains(r'(?i)^MN9|mn9|proboscis')]
print(f"MN9-looking neurons: {len(mn)}")
display(mn[['bodyId', 'type', 'cls', 'nt_used']])

### Cell 3: check 1, direct targets

Subnetwork = JO-C/E plus every neuron they synapse onto directly. Stimulate JO-C/E at 150 Hz for 1 s (5 trials), plus a no-stimulus baseline.

**Pass:** baseline is silent (the model has no spontaneous firing), JO-C/E fire near 150 Hz, and targets with many synapses from JO-C/E fire. If strongly connected targets stay silent, it's a bug, not biology.

In [ ]:
tgt = ws.partners(JO, 'out', min_syn=1).drop(JO, errors='ignore')
print(f"direct targets of JO-C/E: {len(tgt):,}")

sub1 = ws.subnetwork(np.concatenate([JO, tgt.index]))
net1 = LIFNet(sub1, stim_global=JO)

spk0 = net1.run(rate=0 * Hz, n_trials=2)
print(f"baseline spikes: {len(spk0)} (should be 0)\n")

spk1 = net1.run(n_trials=5)
r1 = net1.rates(spk1, ws)
r1['syn_from_JO'] = tgt.reindex(r1.index)

t1 = r1.loc[tgt.index]
print(f"\nJO-C/E mean rate: {r1.loc[JO, 'rate_hz'].mean():.0f} Hz (stimulus 150 Hz)")
for k in [1, 5, 10, 20, 50]:
    s = t1[t1.syn_from_JO >= k]
    print(f"targets with >= {k:2d} synapses from JO: {len(s):5,}, firing >= 1 Hz: {(s.rate_hz >= 1).mean():6.1%}")
print(f"rank correlation, synapses from JO vs rate: {t1.syn_from_JO.corr(t1.rate_hz, method='spearman'):.2f}")
display(t1.sort_values('syn_from_JO', ascending=False).head(20))

### Cell 4: raster and rate vs connection strength

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))

show = np.concatenate([JO[:40], t1.sort_values('syn_from_JO', ascending=False).index[:60]])
local = pd.Series(np.arange(len(net1.keep)), index=net1.keep).loc[show].to_numpy()
row = {l: r for r, l in enumerate(local)}
s = spk1[(spk1.trial == 0) & spk1.local.isin(local)]
ax[0].scatter(s.t, s.local.map(row), s=1, c='k')
ax[0].axhline(39.5, color='tab:red', lw=0.8)
ax[0].set(xlabel='time (s)', ylabel='neuron', title='Trial 0: 40 JO-C/E (below line), top 60 targets (above)')

ax[1].scatter(t1.syn_from_JO, t1.rate_hz, s=6, alpha=0.5)
ax[1].set(xscale='log', xlabel='synapses from JO-C/E', ylabel='rate (Hz)', title='Direct targets')
plt.tight_layout()
plt.savefig(DATA / 'check1_direct_targets.png', dpi=120)